# Семинар 5. Закон больших чисел и центральная предельная теорема

**Курс «Теория вероятностей», ИТМО.** Лекция 5: связь случайности и статистики, поведение выборочных характеристик, влияние размера выборки.

Сегодня увидим, как случайность отдельных наблюдений превращается в устойчивость средних, проверим ЦПТ на суммах и на реальных данных, научимся считать погрешность и нужный объём выборки и разберём ситуации, когда стандартные формулы врут.

| Время | Блок | Что делаем |
|---|---|---|
| 0–5 мин | 0 | Запускаем ноутбук |
| 5–25 мин | 1 | ЗБЧ: Чебышёв против реальности, ошибка игрока, Монте-Карло |
| 25–45 мин | 2 | Выборочные характеристики: выборочное распределение, бутстреп, реальные данные |
| 45–65 мин | 3 | ЦПТ: суммы кубиков, Муавр — Лаплас, скорость сходимости |
| 65–80 мин | 4 | Размер выборки: интервалы, покрытие, «самая опасная формула» |
| 80–90 мин | 5 | Бутстреп: интервал для медианы, где бутстреп ломается |

Задания отмечены **✏️ Задание** и строкой `# TODO` в коде. Там, где стоит `...`, нужно дописать код.

## Как открыть ноутбук в Google Colab

1. Скачайте файл `05-seminar-lln-clt-itmo.ipynb` на свой компьютер.
2. Откройте [colab.research.google.com](https://colab.research.google.com) и войдите в свой Google-аккаунт.
3. Выберите «Файл → Загрузить блокнот» (File → Upload notebook), перейдите на вкладку «Загрузить» и выберите скачанный файл. Можно просто перетащить файл в окно.
4. Ноутбук сохранится на вашем Google Диске в папке «Colab Notebooks». В следующий раз открывайте его оттуда, а не загружайте заново, иначе потеряете свои правки.
5. «Среда выполнения → Выполнить все» (Runtime → Run all) запускает все ячейки; одна ячейка запускается сочетанием `Shift+Enter`.

Ничего устанавливать не нужно: `numpy`, `scipy`, `pandas` и `matplotlib` в Colab уже есть.

## 0. Подготовка

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats

rng = np.random.default_rng(2026)   # фиксируем генератор, чтобы результаты повторялись
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
print("numpy", np.__version__, "| scipy", scipy.__version__)

**Памятка.**

| Что | Код |
|---|---|
| $\Phi(x)$, $1-\Phi(x)$, квантиль $z_q$ | `stats.norm.cdf(x)`, `stats.norm.sf(x)`, `stats.norm.ppf(q)` |
| биномиальное: $P(S\le k)$, $P(S>k)$ | `stats.binom.cdf(k, n, p)`, `stats.binom.sf(k, n, p)` |
| скользящее среднее | `np.cumsum(x) / np.arange(1, len(x) + 1)` |
| стандартная ошибка среднего | `x.std(ddof=1) / np.sqrt(len(x))` |
| QQ-график | `stats.probplot(x, dist="norm", plot=plt)` |

---
## 1. Закон больших чисел

### 1.1. Частота стремится к вероятности

Четыре серии по 10 000 бросков монеты. Частота орлов сначала сильно «гуляет», потом прижимается к $1/2$.

In [ ]:
n = np.arange(1, 10_001)
for _ in range(4):
    flips = rng.random(n.size) < 0.5
    plt.semilogx(n, np.cumsum(flips) / n, lw=1)
plt.fill_between(n, 0.5 - 2 * np.sqrt(0.25 / n), 0.5 + 2 * np.sqrt(0.25 / n), alpha=0.1, label="0,5 ± 2·√(p(1−p)/n)")
plt.axhline(0.5, color="k", ls="--"); plt.ylim(0, 1)
plt.xlabel("n"); plt.ylabel("частота орлов"); plt.legend(); plt.show()

### ✏️ Задание 1.1. Чебышёв против реальности

Для $n\in\{10, 100, 1000, 10\,000\}$ найдите вероятность $P\big(|S_n/n-0{,}5|\ge0{,}05\big)$ тремя способами:

1. **точно** через `stats.binom` (подсказка: $|S_n-n/2|\ge0{,}05n$ означает $S_n\le0{,}45n$ или $S_n\ge0{,}55n$; аккуратно с границами — это целые числа);
2. **оценкой Чебышёва** $\min\big(1,\ \frac{p(1-p)}{n\varepsilon^2}\big)$;
3. **по ЦПТ** без поправки: $2\big(1-\Phi(\varepsilon\sqrt n/\sigma)\big)$, $\sigma=1/2$.

Во сколько раз Чебышёв завышает вероятность при $n=1000$?

In [ ]:
p, eps = 0.5, 0.05
rows = []
for n in [10, 100, 1000, 10_000]:
    # TODO
    exact = ...
    cheb = ...
    clt = ...
    rows.append((n, exact, cheb, clt))
pd.DataFrame(rows, columns=["n", "точно", "Чебышёв", "ЦПТ"]).round(5)

### ✏️ Задание 1.2. Ошибка игрока

Сгенерируйте 1 000 000 бросков честной монеты. Найдите все моменты, когда **пять предыдущих** бросков подряд были решками, и посчитайте долю орлов в **следующем** броске. «Должны» ли орлы чаще выпадать после серии решек?

Отдельно посмотрите на разность «орлы − решки» после $n$ бросков: уменьшается ли она по модулю с ростом $n$?

In [ ]:
flips = rng.random(1_000_000) < 0.5        # True = орёл

# TODO: доля орлов сразу после пяти решек подряд
after_streak = ...
print(f"после 5 решек подряд орёл выпал в доле {after_streak:.4f} случаев")

# TODO: |орлы − решки| после n = 100, 10 000, 1 000 000 бросков
...

### ✏️ Задание 1.3. Монте-Карло с погрешностью

Интеграл $I=\int_0^1 e^{-x^2}\,dx$ не берётся в элементарных функциях. Оцените его методом Монте-Карло: $I=E\,e^{-U^2}$, $U\sim U(0;1)$.

1. Для $n=10^4$ найдите оценку `I_hat` и её стандартную ошибку `se` $=s/\sqrt n$; постройте интервал $\hat I\pm1{,}96\,se$.
2. Сравните с точным значением `quad`. Попало ли оно в интервал?
3. Сколько точек нужно, чтобы погрешность (полуширина интервала 95 %) была не больше $10^{-4}$? Используйте оценку $s$ из п. 1.
4. Повторите п. 1 500 раз и посчитайте, в какой доле повторов интервал накрыл точное значение.

In [ ]:
from scipy.integrate import quad
I_true, _ = quad(lambda x: np.exp(-x**2), 0, 1)

n = 10_000
# TODO
I_hat = ...
se = ...
n_needed = ...
coverage = ...

print(f"I = {I_true:.6f},  оценка {I_hat:.5f} ± {1.96 * se:.5f}")
print(f"нужно точек для ±1e-4: {n_needed},  покрытие интервала: {coverage:.3f}")

---
## 2. Выборочные характеристики

### ✏️ Задание 2.1. Выборочное распределение среднего и медианы

Генеральная совокупность — $\mathrm{Exp}$ со средним 10 (время ожидания в минутах). Выборка объёма $n=25$.

1. Сгенерируйте 100 000 выборок (матрица `100_000 × 25`), посчитайте для каждой среднее и медиану.
2. Найдите стандартные ошибки обеих статистик по симуляции и сравните с теорией: $SE(\bar X)=\sigma/\sqrt n$, $SE(\text{мед.})\approx\dfrac{1}{2f(m)\sqrt n}$, где $m=10\ln2$ — медиана распределения, $f$ — плотность.
3. Чему в среднем равна выборочная медиана? Почему она оценивает не $EX=10$?
4. Нарисуйте гистограммы обеих статистик на одном графике.

In [ ]:
n, mu = 25, 10
S = rng.exponential(mu, size=(100_000, n))

# TODO
means = ...
medians = ...
se_mean_sim, se_mean_theory = ..., ...
se_med_sim, se_med_theory = ..., ...

print(f"SE среднего: симуляция {se_mean_sim:.3f}, теория {se_mean_theory:.3f}")
print(f"SE медианы:  симуляция {se_med_sim:.3f}, теория {se_med_theory:.3f}")

### ✏️ Задание 2.2. Бутстреп: стандартная ошибка по одной выборке

В жизни у нас **одна** выборка, а не 100 000. Бутстреп заменяет неизвестное распределение эмпирическим: извлекаем из выборки $B$ новых выборок того же объёма **с возвращением** и смотрим на разброс статистики.

1. Возьмите одну выборку `x = S[0]`.
2. Сделайте $B=10\,000$ бутстреп-выборок (`rng.choice(x, size=(B, n), replace=True)`), найдите `se_boot_median` — стандартное отклонение медиан.
3. Сравните с «настоящей» SE медианы из задания 2.1. Повторите для нескольких разных исходных выборок: насколько стабильна бутстреп-оценка при $n=25$?

In [ ]:
B = 10_000
for i in range(5):
    x = S[i]
    # TODO
    se_boot_median = ...
    print(f"выборка {i}: бутстреп SE медианы = {se_boot_median:.3f}   (истинная ≈ {se_med_sim:.3f})")

### ✏️ Задание 2.3. Реальная генеральная совокупность

Возьмём датасет California Housing (около 17 000 районов) и будем считать его **всей генеральной совокупностью**. Тогда истинное среднее `median_house_value` нам известно, и можно честно проверить, как ведут себя выборки.

1. Найдите истинные $\mu$ и $\sigma$ по всему датасету.
2. Для $n\in\{5, 30, 200\}$ извлеките 10 000 случайных выборок **без возвращения** (по районам) и постройте гистограммы выборочных средних. Сравните их разброс с $\sigma/\sqrt n$.
3. Для $n=30$ постройте QQ-график выборочных средних. Нормально ли их распределение, хотя само `median_house_value` сильно скошено и обрезано на 500 001?
4. Для $n=30$ проверьте, в какой доле выборок интервал $\bar x\pm1{,}96\,s/\sqrt n$ накрывает истинное $\mu$.

In [ ]:
import os

PATHS = ["/content/sample_data/california_housing_train.csv",
         "https://download.mlcc.google.com/mledu-datasets/california_housing_train.csv"]
df = None
for path in PATHS:
    try:
        if path.startswith("http") or os.path.exists(path):
            df = pd.read_csv(path)
            print("загружено:", path)
            break
    except Exception as exc:
        print("не получилось:", path, exc)
if df is None:   # запасной вариант без интернета: синтетические данные похожей формы
    print("Датасет недоступен, используем синтетические данные")
    v = np.clip(rng.lognormal(np.log(180_000), 0.55, 17_000), 15_000, 500_001)
    df = pd.DataFrame({"median_house_value": np.round(v)})

value = df["median_house_value"].to_numpy(float)
plt.hist(value, bins=80); plt.title("median_house_value: вся совокупность"); plt.show()

In [ ]:
# TODO: 1–4
mu_pop, sigma_pop = ..., ...

---
## 3. Центральная предельная теорема

### 3.1. Суммы разных слагаемых

Возьмём 60 000 сумм по $n$ слагаемых, стандартизуем и сравним с $N(0,1)$.

In [ ]:
gens = {"U(0;1)": (lambda size: rng.random(size), 0.5, np.sqrt(1 / 12)),
        "Exp(1)": (lambda size: rng.exponential(size=size), 1, 1),
        "Lognormal(0;1)": (lambda size: rng.lognormal(0, 1, size), np.exp(0.5), np.sqrt((np.e - 1) * np.e))}
z = np.linspace(-4, 4, 200)
fig, axs = plt.subplots(3, 4, figsize=(14, 7))
for i, (name, (gen, mu, sd)) in enumerate(gens.items()):
    for j, n in enumerate([1, 3, 10, 50]):
        Z = (gen((60_000, n)).sum(1) - n * mu) / (sd * np.sqrt(n))
        axs[i, j].hist(Z, bins=np.linspace(-4, 6, 100), density=True, alpha=0.7)
        axs[i, j].plot(z, stats.norm.pdf(z)); axs[i, j].set_title(f"{name}, n = {n}", fontsize=10)
plt.tight_layout(); plt.show()

*Вопрос.* Для какого распределения нормальное приближение при $n=50$ хуже всего и почему? Посчитайте асимметрию трёх распределений (`stats.skew` по большой выборке).

### ✏️ Задание 3.1. Сумма кубиков: точно против ЦПТ

$S$ — сумма очков на $n=10$ кубиках.

1. Найдите **точное** распределение $S$ свёрткой: начните с `pmf = np.ones(6) / 6` и 9 раз примените `np.convolve`. Значения $S$ — от 10 до 60.
2. Найдите точно $P(S\ge45)$ и сравните с ЦПТ без поправки и с поправкой на непрерывность ($ES=35$, $DS=10\cdot35/12$).
3. Нарисуйте точное распределение и плотность нормального приближения на одном графике.

In [ ]:
n = 10
# TODO
pmf = ...
values = np.arange(n, 6 * n + 1)
p_exact = ...
p_clt = ...
p_clt_cc = ...
print(f"точно {p_exact:.5f},  ЦПТ {p_clt:.5f},  ЦПТ с поправкой {p_clt_cc:.5f}")

### ✏️ Задание 3.2. Муавр — Лаплас: когда работает, а когда нет

Для $S\sim B(n,p)$ найдите $P(S\le np)$ точно и по нормальному приближению с поправкой ($P\approx\Phi\big((\lfloor np\rfloor+0{,}5-np)/\sqrt{np(1-p)}\big)$) для таблицы значений: $p\in\{0{,}5;\ 0{,}1;\ 0{,}01\}$, $n\in\{20, 100, 1000\}$. Добавьте столбец с приближением Пуассона `stats.poisson.cdf(floor(np), n*p)`.

В каких строках ошибка нормального приближения больше 0,01? Совпадает ли это с правилом $np(1-p)\ge10$?

In [ ]:
rows = []
for p in [0.5, 0.1, 0.01]:
    for n in [20, 100, 1000]:
        k = np.floor(n * p)
        # TODO
        exact = ...
        normal = ...
        poisson = ...
        rows.append((p, n, n * p * (1 - p), exact, normal, poisson))
pd.DataFrame(rows, columns=["p", "n", "np(1-p)", "точно", "нормальное", "Пуассон"]).round(4)

---
## 4. Размер выборки

### ✏️ Задание 4.1. Покрытие интервала для доли

Интервал $\hat p\pm1{,}96\sqrt{\hat p(1-\hat p)/n}$ называют интервалом Вальда. Проверьте, в какой доле случаев он накрывает истинное $p$, для $p\in\{0{,}5;\ 0{,}1;\ 0{,}02\}$ и $n\in\{20, 100, 1000\}$ (по 100 000 повторов через `rng.binomial`).

Где покрытие сильно меньше 95 %? Что происходит, когда $\hat p=0$?

In [ ]:
rows = []
for p in [0.5, 0.1, 0.02]:
    for n in [20, 100, 1000]:
        k = rng.binomial(n, p, size=100_000)
        # TODO
        coverage = ...
        rows.append((p, n, coverage))
pd.DataFrame(rows, columns=["p", "n", "покрытие"]).pivot(index="p", columns="n", values="покрытие").round(3)

### ✏️ Задание 4.2. Сколько опрашивать

1. Сколько человек нужно опросить, чтобы с вероятностью 95 % оценить долю сторонников с погрешностью не больше ±2 п. п.? Посчитайте для худшего случая $p=0{,}5$ и для $p=0{,}1$.
2. Время ответа сервиса имеет $\sigma\approx80$ мс. Сколько запросов `n_resp` нужно замерить, чтобы с вероятностью 95 % оценить среднее время с погрешностью не больше ±5 мс?
3. Проверьте п. 2 симуляцией: время ответа показательное со средним 80 мс (у него и $\sigma=80$). Сделайте 10 000 выборок объёма `n_resp` и найдите долю случаев `share_within`, когда $|\bar x-80|\le5$.

In [ ]:
# TODO
n_poll_05 = ...
n_poll_01 = ...
n_resp = ...
share_within = ...
print(n_poll_05, n_poll_01, n_resp, share_within)

### ✏️ Задание 4.3. «Самая опасная формула»: рейтинг школ

Ниже 1 000 школ разного размера. У **всех** школ одинаковая вероятность, что выпускник сдаст экзамен на «отлично», $p=0{,}15$, кроме пяти школ, где она действительно выше ($p=0{,}22$).

1. Найдите долю отличников `rate` в каждой школе. Какого размера школы попали в топ-20 и в последние 20 рейтинга? Сравните медианный размер этих школ с медианой по всем.
2. Посчитайте $z$-оценки $z_i=\dfrac{\hat p_i-\bar p}{\sqrt{\bar p(1-\bar p)/n_i}}$, где $\bar p$ — общая доля по всем выпускникам. Какие школы имеют $z>3{,}09$ (верхняя граница воронки 99,9 %)? Есть ли среди них «настоящие» сильные школы?
3. Постройте воронку: по оси $x$ — размер школы (лог-шкала), по $y$ — доля, плюс границы $\bar p\pm1{,}96\sqrt{\bar p(1-\bar p)/n}$ и $\pm3{,}09\sqrt{\dots}$.

In [ ]:
n_school = np.round(np.exp(rng.uniform(np.log(15), np.log(800), 1000))).astype(int)
p_true = np.full(1000, 0.15)
strong = rng.choice(np.where(n_school > 300)[0], 5, replace=False)
p_true[strong] = 0.22
excellent = rng.binomial(n_school, p_true)
schools = pd.DataFrame({"n": n_school, "excellent": excellent, "strong": np.isin(np.arange(1000), strong)})
schools.head()

In [ ]:
# TODO: 1–3

---
## 5. Бутстреп

В задании 2.2 бутстреп оценивал стандартную ошибку. Теперь построим с его помощью интервал, проверим, насколько он честный, и найдём случаи, где он ошибается.

**Алгоритм.** По выборке $x_1,\dots,x_n$ и статистике $T$:
1. Извлечь из $x$ **с возвращением** $n$ значений, это бутстреп-выборка $x^*$.
2. Посчитать $T^*=T(x^*)$.
3. Повторить $B$ раз (обычно $B=2\,000$–$10\,000$).
4. $SE_{\text{boot}}$ — стандартное отклонение чисел $T^*_1,\dots,T^*_B$; **процентильный интервал 95 %** — их квантили 2,5 % и 97,5 %.

Быстрый способ в `numpy`: `idx = rng.integers(0, n, size=(B, n))`, затем `x[idx]` — сразу $B$ бутстреп-выборок по строкам.

### ✏️ Задание 5.1. Интервал для медианы и его покрытие

Время на сайте распределено логнормально: $\ln X\sim N(4, 1)$ (секунды), истинная медиана $e^4\approx54{,}6$ с. Формулы для SE медианы по выборке у нас нет, а бутстреп её не требует.

1. Для выборки `x` объёма $n=60$ постройте процентильный бутстреп-интервал 95 % для медианы (`ci_median`, $B=2\,000$).
2. Проверьте покрытие: 1 000 раз возьмите новую выборку из генеральной совокупности, постройте для неё такой же интервал и посчитайте долю `coverage_boot` интервалов, накрывших $e^4$.
3. Для сравнения найдите покрытие `coverage_clt` интервала $\bar x\pm1{,}96\,s/\sqrt n$ для **среднего** (истинное среднее $e^{4{,}5}\approx90{,}0$). Почему оно ниже 0,95?

In [ ]:
n, B = 60, 2_000
true_median, true_mean = np.exp(4), np.exp(4.5)
x = rng.lognormal(4, 1, n)

def boot_ci_median(x, B=2_000):
    # TODO: процентильный интервал 95 % для медианы
    ...

# TODO
ci_median = ...
coverage_boot = ...
coverage_clt = ...
print(ci_median, coverage_boot, coverage_clt)

### ✏️ Задание 5.2. Где бутстреп ломается: максимум

Наблюдения равномерны на $[0;\theta]$, $\theta=1$, $n=50$. Оцениваем $\theta$ максимумом выборки.

1. Сделайте $B=10\,000$ бутстреп-выборок и найдите долю `share_max_equal` таких, у которых максимум **совпал** с максимумом исходной выборки.
2. Сравните с теорией: максимум не попадает в бутстреп-выборку с вероятностью $(1-1/n)^n$. Посчитайте `share_theory` $=1-(1-1/n)^n$. К чему это стремится при $n\to\infty$?
3. Постройте гистограммы настоящего распределения $\theta-\max$ (симуляция по 10 000 новым выборкам) и бутстреп-распределения $\max-\max^*$. Почему у второй огромный столбик в нуле?

In [ ]:
n, B = 50, 10_000
x = rng.uniform(0, 1, n)

# TODO
share_max_equal = ...
share_theory = ...
print(share_max_equal, share_theory)

### ✏️ Задание 5.3. Где бутстреп ломается: зависимые наблюдения

Обычный бутстреп перемешивает наблюдения и тем самым **разрушает зависимость**. Возьмём ряд $x_t=0{,}8\,x_{t-1}+\varepsilon_t$, $\varepsilon_t\sim N(0,1)$, длины $n=500$ (как дневные продажи или температура).

1. Оцените SE среднего обычным бутстрепом (`se_iid`).
2. Найдите настоящую SE (`se_true`): 2 000 раз сгенерируйте новый ряд и возьмите разброс средних.
3. **Блочный бутстреп**: разрежьте ряд на 20 подряд идущих блоков по 25 наблюдений, выбирайте с возвращением блоки и склеивайте их. Найдите `se_block`.
4. Во сколько раз обычный бутстреп занижает SE? Сравните с формулой $n_{\text{eff}}\approx n\frac{1-\varphi}{1+\varphi}$ из лекции.

In [ ]:
def ar1(rng, n=500, phi=0.8):
    e = rng.normal(size=n)
    x = np.empty(n)
    x[0] = e[0] / np.sqrt(1 - phi ** 2)
    for t in range(1, n):
        x[t] = phi * x[t - 1] + e[t]
    return x

x = ar1(rng)

# TODO
se_iid = ...
se_true = ...
se_block = ...
print(se_iid, se_true, se_block)

---
## Итоги

- ЗБЧ: частота сходится к вероятности, среднее — к ожиданию; Чебышёв это гарантирует, но числовые оценки даёт ЦПТ.
- Отклонения не «компенсируются», а разбавляются: ошибка игрока.
- Метод Монте-Карло — это ЗБЧ, а его погрешность $s/\sqrt n$ — это ЦПТ.
- У каждой статистики есть выборочное распределение; его разброс можно найти формулой, симуляцией или бутстрепом.
- ЦПТ работает быстро для симметричных слагаемых и медленно для скошенных; для биномиального нужно $np(1-p)\gtrsim10$ и поправка ±0,5.
- Интервал $\bar x\pm1{,}96\,s/\sqrt n$ и объём выборки $n\ge(z\sigma/\varepsilon)^2$; при малых $np$ интервал Вальда врёт.
- Крайние значения рейтинга — у малых выборок; смотрите на воронку.
- Бутстреп даёт SE и интервал для любой статистики по одной выборке, но не работает для максимума и для зависимых данных без блоков.
- Наблюдения должны быть независимыми: считайте погрешность по независимым единицам.

**Домашнее задание** — общий ноутбук по лекциям 4 и 5 `homework/hw04-05/hw04-05-expectation-lln-clt-itmo.ipynb` (условия и порядок сдачи — внутри).